# Advanced: Multi-Strategy Categorical Generalization with PAMOLA.CORE

**Goal**: Master all 3 categorical anonymization strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Hierarchy-based generalization with custom dictionaries
- **Strategy 2**: Frequency-based category grouping
- **Strategy 3**: Merge low-frequency categories
- Calculate advanced privacy metrics (k-anonymity, l-diversity)
- Analyze information loss
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern

---

## Step 1: Setup and Imports

**How to use:**
- Import all required libraries
- Set up paths and configurations
- Verify PAMOLA installation

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Get current notebook location
current_dir = Path.cwd()
print(f"📍 Notebook location: {current_dir}")

# Navigate up to find project root (where pamola_core exists)
project_root = current_dir

# Go up maximum 6 levels to find pamola_core
for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError(
        f"❌ Could not find pamola_core directory!\n"
        f"Searched up to: {project_root}\n"
        f"Current location: {current_dir}"
    )

# Add to Python path
sys.path.insert(0, str(project_root))
print(f"✓ Added to sys.path\n")

# Import PAMOLA modules
print("📦 Importing PAMOLA modules...")
from pamola_core.anonymization.generalization.categorical_op import (
    CategoricalGeneralizationOperation
)
from pamola_core.utils.ops.op_data_source import DataSource
from pamola_core.utils.progress import HierarchicalProgressTracker
from pamola_core.utils.tasks.task_reporting import TaskReporter

print("✅ All imports successful!\n")
print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print(f"📂 Project root:  {project_root.name}")
print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
print("=" * 80)

## Step 2: Load Complex Dataset

**How to use:**
- Load larger dataset from examples/data_examples/category_hierarchy_data.csv
- Or generate realistic 1000-record sample
- Inspect data structure and distributions

**Dataset features:**
- 1000 records
- Multiple sensitive fields
- Hierarchical relationships (categories, locations)

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'category_hierarchy_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = pd.read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Categories with hierarchy
    categories = [
        'Type A1', 'Type A2', 'Type A3',
        'Type B1', 'Type B2',
        'Type C1', 'Type C2', 'Type C3',
        'Type D1', 'Type D2',
        'Type E1', 'Type E2', 'Type E3',
        'Type F1', 'Type F2', 'Type F3',
        'Type G1', 'Type G2', 'Type H1', 'Type H2'
    ]
    
    # Cities with geographic hierarchy
    cities = [
        'City A1', 'City A2', 'City A3',
        'City B1', 'City B2', 'City B3',
        'City C1', 'City C2', 'City C3',
        'City D1', 'City D2', 'City D3',
        'City E1', 'City E2', 'City E3'
    ]
    
    groups = ['Group A', 'Group B', 'Group C', 'Group D', 'Group E', 'Group F']
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'category': np.random.choice(categories, n_records),
        'location': np.random.choice(cities, n_records),
        'age': np.random.randint(18, 80, n_records),
        'value': np.random.randint(10, 500, n_records),
        'group': np.random.choice(groups, n_records)
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if df[col].dtype == 'object':
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Hierarchy Dictionary

**How to use:**
- Check if hierarchy dictionary already exists
- If not found, create example hierarchy dictionary
- Dictionary will be used in Strategy 1 (hierarchy-based)

**Hierarchy levels:**
- **Level 0**: Original (e.g., "Type A1")
- **Level 1**: Specific (e.g., "Type A")
- **Level 2**: General (e.g., "Type")
- **Level 3**: Very General (e.g., "General")

**Note:** You can replace this with your own hierarchy dictionary!

In [ ]:
# Define hierarchy path (in examples directory)
examples_dir = project_root / 'examples'
hierarchy_path = examples_dir / 'data_examples' / 'category_hierarchy.json'

# Check if hierarchy file already exists
if hierarchy_path.exists():
    print(f"✓ Found existing hierarchy dictionary: {hierarchy_path}")
    print(f"📂 Using existing file for Strategy 1\n")
    
    # Load and display info
    with open(hierarchy_path, 'r') as f:
        existing_hierarchy = json.load(f)
    
    print("📊 Existing Dictionary Info:")
    print(f"  Format:   v{existing_hierarchy.get('format_version', 'N/A')}")
    print(f"  Type:     {existing_hierarchy.get('hierarchy_type', 'N/A')}")
    print(f"  Levels:   {existing_hierarchy.get('levels', 'N/A')}")
    print(f"  Mappings: {len(existing_hierarchy.get('data', {}))} entries")
    
    print("\n💡 To use a different hierarchy, replace the file or rename it.")
    print("=" * 80)

else:
    print("⚠️  Hierarchy dictionary not found!")
    print("🔧 Creating example hierarchy dictionary...\n")
    print("=" * 80)

    # Define hierarchy structure (new format with levels and data)
    category_hierarchy = {
        "format_version": "1.0",
        "hierarchy_type": "categorical",
        "description": "Hierarchical generalization for Type categories",
        "levels": ["specific", "general", "very_general"],
        "data": {}
    }
    
    # Generate mappings for all Type categories
    type_letters = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    for letter in type_letters:
        for num in ['1', '2', '3']:
            key = f"Type {letter}{num}"
            # Skip non-existent ones based on generated data
            if letter == 'B' and num == '3':
                continue
            if letter == 'H' and num == '3':
                continue
            category_hierarchy["data"][key] = {
                "specific": f"Type {letter}",
                "general": "Type",
                "very_general": "General"
            }

    # Save to file (with error handling)
    try:
        hierarchy_path.parent.mkdir(parents=True, exist_ok=True)
        with open(hierarchy_path, 'w') as f:
            json.dump(category_hierarchy, f, indent=2)
        print(f"✓ Example hierarchy dictionary created: {hierarchy_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {hierarchy_path} (file may be open)")
        print(f"   Hierarchy exists in memory only")

    print(f"\n📊 Statistics:")
    print(f"  Format version: {category_hierarchy['format_version']}")
    print(f"  Hierarchy type: {category_hierarchy['hierarchy_type']}")
    print(f"  Total mappings: {len(category_hierarchy['data'])}")
    print(f"  Hierarchy depth: {len(category_hierarchy['levels'])} levels")
    print(f"  Level names: {', '.join(category_hierarchy['levels'])}")

    print(f"\n🌳 Example Hierarchy:")
    print(f"  Level 0 (Original):      Type A1")
    print(f"  Level 1 (Specific):      Type A")
    print(f"  Level 2 (General):       Type")
    print(f"  Level 3 (Very General):  General")

    print("\n💡 You can edit this file to match your data categories!")
    print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
- Create shared task directory
- Define config helper function (production pattern)
- Setup DataSource and Reporter
- **Configure field names** for each strategy

**This setup will be reused for all 3 strategies**

**Field configuration:**
```python
FIELD_CONFIG = {
    "strategy1_field": "category",   # For hierarchy strategy
    "strategy2_field": "location",   # For frequency strategy
    "strategy3_field": "group"       # For merge strategy
}
```

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "category",   # Hierarchy-based generalization
    "strategy2_field": "location",   # Frequency-based generalization
    "strategy3_field": "group"       # Merge low-frequency
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy categorical generalization",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Hierarchy-Based Generalization

**How to use:**
- Uses hierarchy dictionary from external file
- Apply semantic generalization based on domain knowledge
- Preserve meaning through predefined hierarchy levels

**Key parameters:**
- `strategy='hierarchy'`
- `external_dictionary_path`: Path to category_hierarchy.json
- `hierarchy_level=2`: Use "General" level
- `case_sensitive=False`: Ignore case differences

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: HIERARCHY-BASED GENERALIZATION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Hierarchy-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = CategoricalGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    strategy='hierarchy',
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_hierarchy",
    output_format='csv',
    external_dictionary_path=str(hierarchy_path),
    hierarchy_level=1,
    case_sensitive=False,
    group_rare_as='OTHER',
    privacy_check_enabled=True,
    use_vectorization=True,
    use_cache=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_hierarchy',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    generate_visualization=True,
    save_output=True,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results
output_files_s1 = sorted(list((task_dir / 'strategy1_hierarchy' / 'output').glob('*.csv')), 
                         key=lambda x: x.stat().st_mtime, reverse=True)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_hierarchy"
    
    print(f"\n📊 Reduction: {df[field_s1].nunique()} → {result_df_s1[output_field_s1].nunique()} categories")

## STRATEGY 2: Frequency-Based Generalization

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: FREQUENCY-BASED GENERALIZATION")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(total=6, description="Strategy 2: Frequency", unit="steps")

operation_s2 = CategoricalGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    strategy='frequency_based',
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_frequency",
    freq_threshold=0.05,
    max_categories=6,
    group_rare_as='OTHER',
    use_vectorization=True,
    use_cache=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_frequency',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    generate_visualization=True,
    save_output=True,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(list((task_dir / 'strategy2_frequency' / 'output').glob('*.csv')),
                         key=lambda x: x.stat().st_mtime, reverse=True)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_frequency"
    
    print(f"\n📊 Reduction: {result_df_s1[field_s2].nunique()} → {result_df_s2[output_field_s2].nunique()} categories")

## STRATEGY 3: Merge Low-Frequency

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: MERGE LOW-FREQUENCY")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(total=6, description="Strategy 3: Merge", unit="steps")

operation_s3 = CategoricalGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    strategy='merge_low_freq',
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_merged",
    freq_threshold=0.03,
    max_categories=8,
    group_rare_as='OTHER',
    use_vectorization=True,
    use_cache=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_merge',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    generate_visualization=True,
    save_output=True,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(list((task_dir / 'strategy3_merge' / 'output').glob('*.csv')),
                         key=lambda x: x.stat().st_mtime, reverse=True)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_merged"
    
    print(f"\n📊 Reduction: {result_df_s2[field_s3].nunique()} → {result_df_s3[output_field_s3].nunique()} categories")

## Strategy Comparison

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

## DETAILED ARTIFACT REVIEW (All Strategies)

Review artifacts from all 3 strategies with newest files

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_hierarchy', 'Strategy 1: Hierarchy-Based'),
    ('strategy2_frequency', 'Strategy 2: Frequency-Based'),
    ('strategy3_merge', 'Strategy 3: Merge Low-Freq')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = sorted(list(metrics_dir.glob('*.json')), 
                              key=lambda x: x.stat().st_mtime, reverse=True)
        if metrics_files:
            print(f"\n📄 METRICS: {metrics_files[0].name}")
            try:
                with open(metrics_files[0], 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                    
                if 'effectiveness' in metrics:
                    eff = metrics['effectiveness']
                    print(f"   Original: {eff.get('original_unique')} → Anonymized: {eff.get('anonymized_unique')}")
                    print(f"   Reduction: {eff.get('effectiveness_ratio', 0):.2%}")
                    
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                    
                if 'categorical_info_loss' in metrics:
                    info = metrics['categorical_info_loss']
                    print(f"   Info Loss: {info.get('precision_loss', 0):.2%}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Mapping (NEWEST)
    mapping_files = sorted(list(strategy_dir.glob('**/*mapping*.json')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
    if mapping_files:
        print(f"\n📄 MAPPING: {mapping_files[0].name}")
        try:
            with open(mapping_files[0], 'r') as f:
                mapping_data = json.load(f)
                mappings = mapping_data.get('mappings', {})
                changed = sum(1 for k, v in mappings.items() if k != v)
                print(f"   Total: {len(mappings)}, Changed: {changed}, Unchanged: {len(mappings)-changed}")
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(list(viz_dir.glob('*.png')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Calculate Privacy Metrics

**How to use:**
- Calculate k-anonymity for original and anonymized data
- Compare privacy improvements
- Identify vulnerable groups

**Note:** Run this cell AFTER all 3 strategies complete successfully!

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    group_sizes = df.groupby(quasi_identifiers).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

# Check if FIELD_CONFIG exists (strategies completed)
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    
    # Original
    k_orig = calculate_k_anonymity(df, [field_s1, field_s2])
    print(f"📊 ORIGINAL DATA:")
    print(f"   Min k: {k_orig['min_k']}")
    print(f"   Avg k: {k_orig['avg_k']:.2f}")
    print(f"   Vulnerable groups: {k_orig['vulnerable_groups']}")
    
    # After anonymization (check if result_df_s2 exists)
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        quasi_ids_final = [f"{field_s1}", f"{field_s2}"]
        k_final = calculate_k_anonymity(result_df_s2, quasi_ids_final)
        print(f"\n✨ AFTER ANONYMIZATION:")
        print(f"   Min k: {k_final['min_k']} ({k_final['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_final['avg_k']:.2f} ({k_final['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_final['vulnerable_groups']}")
        
        print(f"\n🎯 Dataset is {k_final['min_k']}-anonymous")
    else:
        print("\n⚠️  Anonymization not completed yet - run Strategy 2 first!")
        
except NameError:
    print("⚠️  FIELD_CONFIG not defined!")
    print("   Please run 'Step 4: Setup Shared Environment' cell first")
    print("   Then run all 3 strategy cells before calculating privacy metrics")

## Export Final Data

**How to use:**
- Export final anonymized dataset
- Include all generalized fields
- Save to pamola_datasets folder

**Note:** Run this cell AFTER all 3 strategies complete successfully!

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
except NameError:
    print("❌ FIELD_CONFIG not defined! Run Step 4 first.")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export base directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Hierarchy-Based Generalization
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: HIERARCHY-BASED GENERALIZATION")
    print("=" * 80)
    
    # Create strategy folder
    s1_dir = export_base_dir / 'strategy1_hierarchy'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n📋 Available columns:")
    print(f"  {result_df_s1.columns.tolist()}")
    
    # Select columns to export
    output_col_s1 = f"{field_s1}_hierarchy"
    
    if output_col_s1 in result_df_s1.columns:
        # Export with original and generalized columns
        export_cols_s1 = ['record_id', field_s1, output_col_s1, 'age', 'value', 
                          'location', 'group', 'region', 'risk_score', 'gender']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        # Save to CSV
        output_path_s1 = s1_dir / 'hierarchy_generalized_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Columns: {len(final_df_s1.columns)}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s1}")
        
        # Preview
        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s1.head())
        
        # Statistics
        print(f"\n📈 Generalization Statistics:")
        print(f"   Original unique values: {result_df_s1[field_s1].nunique()}")
        print(f"   Generalized unique values: {result_df_s1[output_col_s1].nunique()}")
        reduction = (1 - result_df_s1[output_col_s1].nunique() / result_df_s1[field_s1].nunique()) * 100
        print(f"   Reduction: {reduction:.1f}%")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found in result_df_s1")
        print(f"   Available columns: {result_df_s1.columns.tolist()}")
else:
    print("\n⚠️  Strategy 1 data not available (result_df_s1 not found)")

# ============================================================================
# STRATEGY 2: Frequency-Based Generalization
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: FREQUENCY-BASED GENERALIZATION")
    print("=" * 80)
    
    # Create strategy folder
    s2_dir = export_base_dir / 'strategy2_frequency'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n📋 Available columns:")
    print(f"  {result_df_s2.columns.tolist()}")
    
    # Select columns to export
    output_col_s1 = f"{field_s1}_hierarchy"
    output_col_s2 = f"{field_s2}_frequency"
    
    if output_col_s2 in result_df_s2.columns:
        # Export with both strategy 1 and strategy 2 results
        export_cols_s2 = ['record_id', field_s1, output_col_s1, 
                          field_s2, output_col_s2, 
                          'age', 'value', 'group', 'region', 'risk_score', 'gender']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        # Save to CSV
        output_path_s2 = s2_dir / 'frequency_generalized_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Columns: {len(final_df_s2.columns)}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s2}")
        
        # Preview
        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s2.head())
        
        # Statistics
        print(f"\n📈 Generalization Statistics:")
        print(f"   Field: {field_s2}")
        print(f"   Original unique values: {result_df_s2[field_s2].nunique()}")
        print(f"   Generalized unique values: {result_df_s2[output_col_s2].nunique()}")
        reduction = (1 - result_df_s2[output_col_s2].nunique() / result_df_s2[field_s2].nunique()) * 100
        print(f"   Reduction: {reduction:.1f}%")
    else:
        print(f"\n❌ Column '{output_col_s2}' not found in result_df_s2")
        print(f"   Available columns: {result_df_s2.columns.tolist()}")
else:
    print("\n⚠️  Strategy 2 data not available (result_df_s2 not found)")

# ============================================================================
# STRATEGY 3: Merge Low-Frequency
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: MERGE LOW-FREQUENCY")
    print("=" * 80)
    
    # Create strategy folder
    s3_dir = export_base_dir / 'strategy3_merge'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n📋 Available columns:")
    print(f"  {result_df_s3.columns.tolist()}")
    
    # Select columns to export
    output_col_s1 = f"{field_s1}_hierarchy"
    output_col_s2 = f"{field_s2}_frequency"
    output_col_s3 = f"{field_s3}_merged"
    
    if output_col_s3 in result_df_s3.columns:
        # Export with all three strategy results
        export_cols_s3 = ['record_id', 
                          field_s1, output_col_s1,
                          field_s2, output_col_s2,
                          field_s3, output_col_s3,
                          'age', 'value', 'region', 'risk_score', 'gender']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        # Save to CSV
        output_path_s3 = s3_dir / 'merged_generalized_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Columns: {len(final_df_s3.columns)}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open): {output_path_s3}")
        
        # Preview
        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s3.head())
        
        # Statistics
        print(f"\n📈 Generalization Statistics:")
        print(f"   Field: {field_s3}")
        print(f"   Original unique values: {result_df_s3[field_s3].nunique()}")
        print(f"   Generalized unique values: {result_df_s3[output_col_s3].nunique()}")
        reduction = (1 - result_df_s3[output_col_s3].nunique() / result_df_s3[field_s3].nunique()) * 100
        print(f"   Reduction: {reduction:.1f}%")
    else:
        print(f"\n❌ Column '{output_col_s3}' not found in result_df_s3")
        print(f"   Available columns: {result_df_s3.columns.tolist()}")
else:
    print("\n⚠️  Strategy 3 data not available (result_df_s3 not found)")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT SUMMARY")
print("=" * 80)
print(f"\n📂 All files saved to: {export_base_dir}")
print(f"\n📁 Folder structure:")
print(f"   ├── strategy1_hierarchy/")
print(f"   │   └── hierarchy_generalized_data.csv")
print(f"   ├── strategy2_frequency/")
print(f"   │   └── frequency_generalized_data.csv")
print(f"   └── strategy3_merge/")
print(f"       └── merged_generalized_data.csv")

if 'elapsed_s1' in locals() and 'elapsed_s2' in locals() and 'elapsed_s3' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"\n⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ Hierarchy-based generalization
- ✅ Frequency-based generalization  
- ✅ Merge low-frequency categories
- ✅ Privacy metrics calculation
- ✅ Multi-stage pipeline

**Next steps:**
- Experiment with different thresholds
- Try your own data
- Combine with other operations